<a href="https://colab.research.google.com/github/olkachalova/geospat/blob/main/leaf_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 131.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.2 MB/s eta 0:00:00
  Attempting uninstall: lxml
    Found existing installation: lxml 5.4.0
    Uninstalling lxml-5.4.0:
      Successfully uninstalled lxml-5.4.0


In [5]:
from ddgs import DDGS

def search_images(keyword, max_results=10):
    with DDGS() as ddgs:
        images = ddgs.images(
            keyword,
            max_results=max_results
        )
        return [img['image'] for img in images]

In [24]:
keyword = "ash leaf"
image_urls = search_images(keyword, 2000)
len(image_urls)

100

In [ ]:
image_urls[96]

'https://img.freepik.com/premium-photo/oak-leaves_1048944-3285819.jpg'

In [8]:
import os
import requests
from urllib.parse import urlparse
import warnings

def download_image(url, folder, custom_name=None, verbose=True):
    # Create the folder if it doesn't exist
    os.makedirs(folder, exist_ok=True)

    # Get the filename from the URL or use the custom name
    if custom_name:
        filename = custom_name
    else:
        filename = os.path.basename(urlparse(url).path)
        if not filename:
            filename = 'image.jpg'  # Default filename if none is found in the URL

    # Ensure the filename has an extension
    if not os.path.splitext(filename)[1]:
        filename += '.jpg'

    filepath = os.path.join(folder, filename)

    # If the file already exists, append a number to make it unique
    base, extension = os.path.splitext(filepath)
    counter = 1
    while os.path.exists(filepath):
        filepath = f"{base}_{counter}{extension}"
        counter += 1

    try:
        # Send a GET request to the URL with a timeout of 10 seconds
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Raises an HTTPError for bad responses

        # Check if the content type is an image
        content_type = response.headers.get('content-type', '')
        if not content_type.startswith('image'):
            if verbose:
                warnings.warn(f"The URL does not point to an image. Content-Type: {content_type}")
            return False

        # Write the image content to the file
        with open(filepath, 'wb') as f:
            f.write(response.content)

        if verbose:
            print(f"Image successfully downloaded: {filepath}")
        return True

    except requests.exceptions.Timeout:
        if verbose:
            warnings.warn(f"Download timed out for URL: {url}")
    except requests.exceptions.HTTPError as e:
        if verbose:
            warnings.warn(f"HTTP error occurred: {e}")
    except requests.exceptions.RequestException as e:
        if verbose:
            warnings.warn(f"An error occurred while downloading the image: {e}")
    except IOError as e:
        if verbose:
            warnings.warn(f"An error occurred while writing the file: {e}")

    return False

In [25]:
from tqdm.notebook import tqdm

for i, url in enumerate(tqdm(image_urls)):
    download_image(url, "./dataset/ash/", f'image{i+200}.jpg', verbose=False)

  0%|          | 0/100 [00:00<?, ?it/s]

In [37]:
import os
import shutil
import random

# Paths
src_dir = "./dataset"
dst_dir = "./leaf_dataset"

# Train/test ratio
train_ratio = 0.75

# Ensure destination directories exist
for split in ["train", "test"]:
    for cls in os.listdir(src_dir):
        src_path = os.path.join(src_dir, cls)
        if os.path.isdir(src_path):
            os.makedirs(os.path.join(dst_dir, split, cls), exist_ok=True)

# Split images
for cls in os.listdir(src_dir):
    src_path = os.path.join(src_dir, cls)
    if not os.path.isdir(src_path):
        continue

    images = [f for f in os.listdir(src_path) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.tiff','.webp'))]
    random.shuffle(images)

    split_idx = int(len(images) * train_ratio)
    train_imgs = images[:split_idx]
    test_imgs = images[split_idx:]

    for img in train_imgs:
        shutil.copy(os.path.join(src_path, img), os.path.join(dst_dir, "train", cls, img))

    for img in test_imgs:
        shutil.copy(os.path.join(src_path, img), os.path.join(dst_dir, "test", cls, img))

print("✅ Leaf dataset complete!")


✅ Dataset split complete!


In [48]:
src = "/content/leaf_dataset"
dst = "/content/drive/MyDrive/leaf_dataset"

shutil.copytree(src, dst)

'/content/drive/MyDrive/leaf_dataset'

In [38]:
from torchvision import datasets, transforms
import torch

transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor()
])

train_set = datasets.ImageFolder(root='./leaf_dataset/train', transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=256, shuffle=True)

test_set = datasets.ImageFolder(root='./leaf_dataset/test', transform=transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False)

print("Classes:", train_set.classes)

Classes: ['ash', 'linden', 'oak']


In [42]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(in_features=28*28*64, out_features=128)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(in_features=128, out_features=3)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.flat(x)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
model

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (flat): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=50176, out_features=128, bias=True)
  (drop): Dropout(p=0.25, inplace=False)
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)

In [43]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.02)

In [46]:
for epoch in range(50):  # Train for 50 epochs
    running_loss = 0.0
    for images, labels in train_loader:
        # Move images and labels to the device
        images = images.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/50], Loss: {running_loss / len(train_loader):.4f}')


Epoch [1/50], Loss: 1.0538
Epoch [2/50], Loss: 1.0530
Epoch [3/50], Loss: 1.0500
Epoch [4/50], Loss: 1.0486
Epoch [5/50], Loss: 1.0411
Epoch [6/50], Loss: 1.0422
Epoch [7/50], Loss: 1.0385
Epoch [8/50], Loss: 1.0377
Epoch [9/50], Loss: 1.0320
Epoch [10/50], Loss: 1.0332
Epoch [11/50], Loss: 1.0301
Epoch [12/50], Loss: 1.0211
Epoch [13/50], Loss: 1.0207
Epoch [14/50], Loss: 1.0875
Epoch [15/50], Loss: 1.0519
Epoch [16/50], Loss: 1.0161
Epoch [17/50], Loss: 1.0131
Epoch [18/50], Loss: 1.0126
Epoch [19/50], Loss: 1.0514
Epoch [20/50], Loss: 1.0234
Epoch [21/50], Loss: 1.0014
Epoch [22/50], Loss: 0.9928
Epoch [23/50], Loss: 1.0057
Epoch [24/50], Loss: 0.9981
Epoch [25/50], Loss: 1.1032
Epoch [26/50], Loss: 1.0546
Epoch [27/50], Loss: 0.9967
Epoch [28/50], Loss: 0.9899
Epoch [29/50], Loss: 0.9936
Epoch [30/50], Loss: 0.9946
Epoch [31/50], Loss: 1.0168
Epoch [32/50], Loss: 1.0462
Epoch [33/50], Loss: 0.9953
Epoch [34/50], Loss: 0.9713
Epoch [35/50], Loss: 0.9776
Epoch [36/50], Loss: 0.9864
E

In [47]:
correct = 0
total = 0
with torch.no_grad():  # Disable gradient calculation for evaluation
    for images, labels in test_loader:
        # Move images and labels to the device
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')

Accuracy: 55.88%
